# 04 · Association testing

**Continue from [03 · Geuvadis distillation](03_geuvadis_distillation.ipynb)** to test your OneK1K-derived SNP weights against the bundled AD and SLE GWAS tables in `data/data-for-the-paper/gwas/`.

Already have the supplied pretrained models? Choose their directory below. You can skip training and distillation and start here after [Getting Started](../README.md#getting-started) and preparing the GWAS inputs in section 2 below.

The repository's TWAS wrapper hands association testing to the bundled **MetaXcan S-PrediXcan** program, with no ctPred tests. Keep `EXECUTE = False` to preview commands. Switch to `True` when you're ready to run both GWASs.


In [ ]:
from pathlib import Path
import sys

root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src/training/run.py").is_file()),
    None,
)
if root is None:
    raise RuntimeError("Open this notebook from inside the ATTICUS repository.")
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "notebooks"))

from tutorial import ROOT, PATHS, require, run, uv, student_draw_count, file_fingerprint

from src.twas.models import open_students, student_file_config

import gzip
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

## 1. Choose your student models

Choose one of the two `STUDENT_DIR` paths below and leave the other commented out: the **pretrained OneK1K models**, or the **models you trained in notebook 03**. TWAS reads the model settings directly from `students.h5`.

For pretrained models, download the complete folder to `models/onek1k/` first. If you used a different output directory in notebook 03, paste that path here.

Results go under `outputs/tutorial/onek1k/twas/<model-directory>/`, with one folder per trait. You can change this with `twas_root` below.

In [ ]:
EXECUTE = False

STUDENT_DIR = ROOT / "models/onek1k"  # Pretrained models
# STUDENT_DIR = PATHS["output"] / "students-probabilistic-het"  # From notebook 03

JOBS = 1
TRAITS = ["ad", "sle"]
WANDB_ARGS = ["--wandb-mode", "disabled"]

models_input = STUDENT_DIR / "students.h5"
model_label = STUDENT_DIR.name
twas_root = PATHS["output"] / "twas" / model_label

print("Models:", models_input)
print("TWAS output root:", twas_root)


In [ ]:
model_files = []
model_genes = None
if models_input.is_file():
    with open_students(models_input) as handle:
        if handle.attrs.get("format_version") != 1:
            raise ValueError("Expected a native students.h5 file.")
        student_config = json.loads(handle["config"].asstr()[()])
        arguments = student_config["arguments"]
        complete = bool(handle.attrs.get("complete", False))
        covariance_complete = bool(handle.attrs.get("covariances_complete", False))
        model_files = student_file_config(handle, models_input).get("student_files", [])
        model_genes = sorted(handle["genes"])
        covariance_paths = ([models_input.parent / path for path in handle["covariance_paths"].asstr()[:].flat]
                            if "covariance_paths" in handle else [])
        missing_covariances = [path for path in covariance_paths if not path.is_file()]
        display(pd.DataFrame([{
            "mode": arguments["mode"],
            "additive": arguments["additive"],
            "genes": len(handle["genes"]),
            "cell_types": len(handle["cell_types"]),
            "LD_reference_individuals": len(handle["sample_ids"]),
            "draws_per_gene_cell_type": student_draw_count(handle),
            "students_complete": complete,
            "covariance_complete": covariance_complete and not missing_covariances,
        }]))
    if not complete or not covariance_complete or missing_covariances:
        message = "Models or covariance are incomplete. Finish notebook 03 or restore the complete model directory."
        if EXECUTE:
            raise RuntimeError(message)
        print(message)
else:
    print("No students.h5 at the selected path. Complete notebook 03 or select an available pretrained model.")
    print("Commands can still be previewed.")


The covariance reference uses all recorded distillation individuals, normally **503 Geuvadis donors**, including those held out for fidelity evaluation.

TWAS reuses the saved weights and covariance, preparing any missing PredictDB exports for MetaXcan.

## 2. Check the bundled GWAS inputs

GWAS tables can be large, so this quick check reads just the header and first record.

Expected columns are `ID`, `ALT`, `REF`, `ES`, and `SE`. `ES` is the signed effect for `ALT` (log odds for a binary trait), and `SE` is its positive standard error. Use GRCh38 coordinates and forward-strand alleles. MetaXcan aligns effect alleles; liftover or strand complementation, if needed, will need to be done separately!

In [ ]:
gwas_directory = ROOT / "data/data-for-the-paper/gwas"
gwas_files = [gwas_directory / f"gwas-{trait}.tsv.gz" for trait in TRAITS]
required_columns = {"ID", "ALT", "REF", "ES", "SE"}
gwas_preview = []
gwas_issues = []

for path in gwas_files:
    columns, first_id, issue = [], None, None
    if not path.is_file():
        issue = "Missing file; download the bundled GWASs using the command below."
    else:
        try:
            with path.open("rb") as handle:
                prefix = handle.read(128)
            if prefix.startswith(b"version https://git-lfs.github.com/spec/v1"):
                issue = "Git LFS pointer, not downloaded GWAS data."
            else:
                with gzip.open(path, "rt") as handle:
                    columns = handle.readline().strip().split("\t")
                    first_record = handle.readline().strip().split("\t")
                if len(columns) != len(set(columns)) or not required_columns.issubset(columns):
                    issue = f"Unexpected columns: {columns}"
                elif len(first_record) != len(columns):
                    issue = "Missing or malformed first GWAS record."
                else:
                    first_id = first_record[columns.index("ID")]
        except (OSError, EOFError, UnicodeError) as error:
            issue = f"Cannot read compressed GWAS data: {error}"
    if issue:
        gwas_issues.append(f"{path.name}: {issue}")
    gwas_preview.append({
        "trait": path.name.removeprefix("gwas-").removesuffix(".tsv.gz"),
        "file": path.name,
        "status": issue or "Header checked",
        "columns": ", ".join(columns),
        "first_ID": first_id,
    })

display(pd.DataFrame(gwas_preview))
if gwas_issues:
    print("From the repository root, retrieve the bundled files with:")
    print("git lfs pull --include='data/data-for-the-paper/gwas/gwas-ad.tsv.gz,data/data-for-the-paper/gwas/gwas-sle.tsv.gz'")
    print("Commands below still show the expected paths in preview mode.")
    if EXECUTE:
        raise ValueError("Prepare valid GWAS files before execution:\n" + "\n".join(gwas_issues))


## 3. Run S-PrediXcan for each trait

Each GWAS gets its own output directory, using the same completed models. `JOBS = 1` starts one MetaXcan process at a time. Have CPU and memory to psare? Increase it to more and you'll have results quite quickly! Default probabilistic models will need 25 MetaXcan runs per cell type and trait.

TWAS uses **all genes in the selected models**. FYI: the lists in `data/data-for-the-paper/tested_genes/` document the manuscript's analyses.

Also: The testable genes within each cell type determine its multiple-testing correction!

In [ ]:
outputs = {trait: twas_root / trait for trait in TRAITS}

if EXECUTE:
    require([models_input, ROOT / "metaxcan/software/SPrediXcan.py", *gwas_files])

for gwas in gwas_files:
    trait = gwas.name.removeprefix("gwas-").removesuffix(".tsv.gz")
    command = uv(
        "twas", "src/twas/run.py",
        "-m", models_input,
        "--gwas", gwas,
        "-o", outputs[trait],
        "--jobs", JOBS,
        *WANDB_ARGS,
        "--wandb-name", f"twas-{trait}-{model_label}",
    )
    run(command, execute=EXECUTE)


## 4. Read the compact association tables

`associations.tsv` has one row per gene/cell type:

| Columns | Meaning |
| --- | --- |
| `gene`, `cell_type` | Model identifiers. |
| `zscore`, `effect_size` | Mean across available probabilistic draws; the single result for deterministic models. |
| `pvalue`, `neg_log10_pvalue` | Equal-weight ACAT over available draw p-values, or the deterministic p-value. The log form preserves extreme tails. |
| `qvalue` | Benjamini–Hochberg correction across tested genes within that cell type. |
| `n_draws_tested`, `status` | How many draws contributed and whether the result was testable. |

**An ACAT p-value is not the p-value of the mean Z-score.** Use the output p-values for significance, and sort by `neg_log10_pvalue` to keep extreme tails distinguishable.

Now have a look at your results!

In [ ]:
manifests = {}
current_model = file_fingerprint(models_input)

for trait, directory in outputs.items():
    manifest_path = directory / "run.json"
    summary_path = directory / "associations.tsv"
    if not manifest_path.is_file() or not summary_path.is_file():
        print(f"{trait}: no completed results yet.")
        continue
    manifest = json.loads(manifest_path.read_text())
    if not manifest.get("complete", False):
        print(f"{trait}: run is incomplete; inspect {manifest_path}.")
        continue
    config = manifest.get("config", {})
    recorded_files = config.get("student_files", []) + config.get("covariance_files", [])
    models_match = (
        current_model is not None and config.get("models") == current_model
        and config.get("student_files", []) == model_files
        and all(file_fingerprint(saved[0]) == saved for saved in recorded_files)
    )
    current_gwas = file_fingerprint(gwas_directory / f"gwas-{trait}.tsv.gz")
    if (not models_match or current_gwas is None or config.get("gwas") != current_gwas
            or model_genes is None or config.get("genes") != model_genes):
        print(f"{trait}: saved results belong to different or missing model/GWAS inputs; use a new TWAS output directory.")
        continue
    manifests[trait] = manifest
    summary = pd.read_csv(summary_path, sep="\t")
    tested = summary.loc[summary["n_draws_tested"] > 0]
    print(f"{trait.upper()}: {len(tested):,} tested gene/cell-type pairs")
    display(tested.sort_values("neg_log10_pvalue", ascending=False).head(10))
